# Install Dependencies

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 2.3 MB/s eta 0:00:00


# Import Required Modules

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (accuracy_score, f1_score, recall_score, # evaluatin metrics
                             precision_score, confusion_matrix,
                             classification_report)

# For array, dataset, and visualizing
import numpy as np

import pandas as pd
from sklearn.preprocessing import LabelEncoder

import re
import pyarabic.araby as araby
import emoji

# Load Dataset

In [ ]:
data = pd.read_excel('ClimaSentAR Dataset.xlsx')

data.drop_duplicates(subset='text', inplace = True)
data.dropna(inplace = True, subset='text')
data.reset_index(drop=True, inplace = True)

data = data[['text', 'sentiment']]

data.columns = ['text', 'sentiment']
print(data['sentiment'].value_counts())

# Preprocessing

In [ ]:
import re
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.stem.isri import ISRIStemmer

# Initialize stemmer
stemmer = ISRIStemmer()

stopwords_list = set()

# Add custom dialect words (example)
custom_stopwords = {"وين", "شو"}
stopwords_list.update(custom_stopwords)

# Negation words to KEEP (do NOT remove)
negation_words = {"لا", "مو", "مش", "ليس", "ما"}

# Remove negations from stopwords
stopwords_list = stopwords_list - negation_words

arabic_pattern = re.compile(r'[\u0600-\u06FF]+')

def clean_text(text):
    text = str(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove mentions
    text = re.sub(r'@\w+', '', text)

    # Remove hashtag symbol only
    text = re.sub(r'#', '', text)

    # Remove RT
    text = re.sub(r'\bRT\b', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove non-Arabic characters
    words = arabic_pattern.findall(text)
    text = ' '.join(words)

    return text.strip()

def preprocess_text(text):
    text = clean_text(text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords (except negation)
    tokens = [w for w in tokens if w not in stopwords_list]

    # Stemming using ISRI
    tokens = [stemmer.stem(w) for w in tokens]

    return tokens

data['tokens'] = data['text'].apply(preprocess_text)

# Optional: reconstructed cleaned text
data['text'] = data['tokens'].apply(lambda x: ' '.join(x))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF vectorizer with n-grams
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    stop_words='english'
)

# Fit and transform the text data
X = tfidf.fit_transform(data['text'])

In [ ]:
# Encode Labels
label_encoder = LabelEncoder()
data['label_encoded'] = label_encoder.fit_transform(data['sentiment'])
num_classes = len(label_encoder.classes_)

# Split Data

In [ ]:
y = data['label_encoded']

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42)

# SVM

In [ ]:
svm_bow = SVC(random_state = 42)
svm_bow.fit(X_train, y_train)

SVC(random_state=42)

In [1]:
pred = svm_bow.predict(X_test)

f1 = f1_score(y_test, pred, average = 'weighted')
print(f'F1 score = {f1:.3f}')

F1 score = 0.214


# Pred on Asad

In [ ]:
asad = pd.read_csv('train_all.csv')
asad['tokens'] = asad['Text'].apply(preprocess_text)

# Optional: reconstructed cleaned text
asad['Text'] = asad['tokens'].apply(lambda x: ' '.join(x))

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-2201979006.py:4: SyntaxWarning: invalid escape sequence '\s'
  asad2['Text'] = asad2['Text'].str.replace("\s+", " ", regex=True)


,Tweet_id,sentiment,text
0,1221884257490042887,neutral,الزعل بيغير ملامحك بيغير نظرة العين بيغير شكلك...
1,1221884400377499651,neutral,@ @ ليس حبا في ايران بقدر ماهو نكايه بترامب وحزبه
2,1221881406168731649,neutral,@ ابي اعرف الحاكم العربي المسلم اشلون ينام ماي...
3,1221882047691657217,neutral,@ @ في الخطاب تبع سليم سعاده حطت عالتويتر شو ق...
4,1221880371773673474,neutral,@ مفيش الكلام ده في الزمن


In [ ]:
train, asad20 = train_test_split(asad,
                               test_size=20000,
                               stratify=asad['sentiment'],
                               random_state=42)

In [ ]:
asad20['sentiment'] = asad20['sentiment'].str.title()
y_test = label_encoder.fit_transform(asad20['sentiment'])

In [ ]:
X_asad = tfidf.fit_transform(asad20['Text'])

In [2]:
pred = svm_bow.predict(X_asad)

f1 = f1_score(y_test, pred, average = 'weighted')
print(f'F1 score = {f1:.3f}')

F1 score = 0.551


# ASTD

In [ ]:
import pandas as pd
df = pd.read_csv('Tweets.txt', delimiter='\t', header=None, names=['text', 'Label'])  # Change delimiter if needed
df.head()

,text,Label
0,بعد استقالة رئيس #المحكمة_الدستورية ننتظر استق...,OBJ
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,POS
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,NEG
3,#الحرية_والعدالة | شاهد الآن: #ليلة_الاتحادية ...,OBJ
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,NEUTRAL


In [ ]:
df['tokens'] = df['text'].apply(preprocess_text)

# Optional: reconstructed cleaned text
df['text'] = df['tokens'].apply(lambda x: ' '.join(x))

In [ ]:
df = df[df['Label']!='OBJ']
df['Label'] = df['Label'].map({
    'NEG': 'Negative',
    'NEUTRAL': 'Neutral',
    'POS': 'Positive'
})
df.head()

,text,Label
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,Positive
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,Negative
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,Neutral
5,#انتخبوا_العرص #انتخبوا_البرص #مرسى_رئيسى #اين...,Neutral
6,امير عيد هو اللي فعلا يتقال عليه ستريكر صريح #...,Positive


In [ ]:
y_test = label_encoder.fit_transform(df['Label'])

In [ ]:
X_astd = tfidf.fit_transform(df['text'])

In [3]:
pred = svm_bow.predict(X_astd)

f1 = f1_score(y_test, pred, average = 'weighted')
print(f'F1 score = {f1:.3f}')

F1 score = 0.173


# SemEval

In [ ]:
semeval = pd.read_csv('SemEval2017-task4-train.subtask-A.arabic.txt', delimiter='\t', header=None, names=['id', 'sentiment', 'text'])
semeval = semeval[['text', 'sentiment']]

semeval['tokens'] = semeval['Text'].apply(preprocess_text)

# Optional: reconstructed cleaned text
semeval['Text'] = semeval['tokens'].apply(lambda x: ' '.join(x))

semeval['sentiment'] = semeval['sentiment'].map({
    'negative': 'Negative',
    'neutral': 'Neutral',
    'positive': 'Positive'
})
semeval.head()

,text,sentiment
0,إجبار أبل على التعاون على فك شفرة اجهزتها http...,Positive
1,RT @20fourMedia: #غوغل تتحدى أبل وأمازون بأجهز...,Positive
2,جوجل تنافس أبل وسامسونج بهاتف جديد https://t.c...,Positive
3,رئيس شركة أبل: الواقع المعزز سيصبح أهم من الطع...,Positive
4,ساعة أبل في الأسواق مرة أخرى https://t.co/dY2x...,Neutral


In [ ]:
y_test = label_encoder.fit_transform(semeval['sentiment'])

In [ ]:
X_semeval = tfidf.fit_transform(semeval['text'])

In [4]:
pred = svm_bow.predict(X_semeval)

f1 = f1_score(y_test, pred, average = 'weighted')
print(f'F1 score = {f1:.3f}')

F1 score = 0.308
